# 🍎 Entrenamiento del Modelo de Predicción Nutricional

Este notebook permite entrenar el modelo de predicción nutricional desde Google Colab.

## Flujo:
1. **Extraer CSV** - Genera dataset desde la BD
2. **Entrenar Modelo** - Entrena LightGBM con el CSV
3. **Ver Métricas** - Obtiene métricas del modelo
4. **Generar Gráficas** - Genera 5 visualizaciones importantes

## ⚙️ Configuración Inicial

In [ ]:
# ============================================
# CONFIGURACIÓN - MODIFICAR SEGÚN TU ENTORNO
# ============================================

# URL del servidor ML (modelo/ml-recomendator)
ML_API_URL = "http://localhost:8001"  # Local
# ML_API_URL = "https://ml-api.tu-dominio.com"  # Producción

print(f"🔗 ML API URL: {ML_API_URL}")

In [ ]:
# Instalar dependencias
!pip install requests -q

import requests
import json
import base64
from IPython.display import Image, display, HTML

def api_call(method, endpoint, data=None):
    """Helper para llamadas a la API."""
    url = f"{ML_API_URL.rstrip('/')}{endpoint}"
    try:
        if method == "GET":
            response = requests.get(url, timeout=300)
        else:
            response = requests.post(url, json=data, timeout=600)
        return response.json() if response.status_code == 200 else {"error": response.text}
    except Exception as e:
        return {"error": str(e)}

print("✅ Dependencias cargadas")

## 🔍 Verificar Estado del Sistema

In [ ]:
# Verificar conexión y estado
status = api_call("GET", "/api/v1/nutritional/status")

if "error" in status:
    print(f"❌ Error conectando: {status['error']}")
    print(f"   Verifica que el servidor esté corriendo en {ML_API_URL}")
else:
    print("✅ Conexión exitosa")
    print(f"   CSV existe: {status.get('csv_exists', False)}")
    print(f"   Registros CSV: {status.get('csv_records', 0)}")
    print(f"   Modelo existe: {status.get('model_exists', False)}")
    print(f"   Gráficas: {status.get('plots_available', [])}")

---
## 📊 Paso 1: Extraer CSV desde la Base de Datos

In [ ]:
# Parámetros de extracción
extract_params = {
    "min_measurements": 2,      # Mínimo mediciones por niño
    "lookback_months": 24,      # Meses hacia atrás
    "include_synthetic": True,  # Incluir datos sintéticos para balancear
    "target_size": 180          # Tamaño objetivo del dataset
}

print("📊 Extrayendo datos de la BD...")
print(f"   Parámetros: {json.dumps(extract_params, indent=2)}")
print()

result = api_call("POST", "/api/v1/nutritional/extract_csv", extract_params)

if "error" in result:
    print(f"❌ Error: {result['error']}")
else:
    print(f"✅ {result.get('message')}")
    print(f"   Registros: {result.get('records_count', 0)}")
    print(f"   Tiempo: {result.get('extraction_time_seconds', 0):.1f}s")
    print(f"   CSV: {result.get('csv_path')}")
    print()
    print("📈 Distribución de clases:")
    for clase, count in result.get('class_distribution', {}).items():
        print(f"   {clase}: {count}")

---
## 🎯 Paso 2: Entrenar el Modelo

In [ ]:
# Parámetros de entrenamiento
train_params = {
    "validation_split": 0.3,    # 30% para validación
    "num_boost_round": 500,     # Máximo iteraciones
    "early_stopping": 50        # Parada temprana
}

print("🎯 Entrenando modelo LightGBM...")
print(f"   Parámetros: {json.dumps(train_params, indent=2)}")
print()
print("⏳ Esto puede tomar unos minutos...")

result = api_call("POST", "/api/v1/nutritional/train", train_params)

if "error" in result:
    print(f"❌ Error: {result['error']}")
else:
    print()
    print("="*50)
    print("✅ MODELO ENTRENADO EXITOSAMENTE")
    print("="*50)
    print(f"   Accuracy:    {result.get('accuracy', 0):.4f}")
    print(f"   F1 Macro:    {result.get('f1_macro', 0):.4f}")
    print(f"   F1 Weighted: {result.get('f1_weighted', 0):.4f}")
    print(f"   Iteraciones: {result.get('num_iterations', 0)}")
    print(f"   Tiempo:      {result.get('training_time_seconds', 0):.1f}s")
    print(f"   Modelo:      {result.get('model_path')}")

---
## 📈 Paso 3: Obtener Métricas Detalladas

In [ ]:
print("📈 Obteniendo métricas del modelo...")

metrics = api_call("GET", "/api/v1/nutritional/metrics")

if "error" in metrics:
    print(f"❌ Error: {metrics['error']}")
elif not metrics.get('model_loaded'):
    print("⚠️ Modelo no cargado. Entrena primero.")
else:
    print()
    print("="*50)
    print("📊 MÉTRICAS DEL MODELO")
    print("="*50)
    print(f"   Accuracy:    {metrics.get('accuracy', 0):.4f}")
    print(f"   F1 Macro:    {metrics.get('f1_macro', 0):.4f}")
    print(f"   F1 Weighted: {metrics.get('f1_weighted', 0):.4f}")
    print()
    print("🔍 Feature Importance (Top 5):")
    fi = metrics.get('feature_importance', {})
    sorted_fi = sorted(fi.items(), key=lambda x: x[1], reverse=True)[:5]
    for name, importance in sorted_fi:
        print(f"   {name}: {importance:.2f}")

In [ ]:
# Métricas por clase
if metrics.get('class_metrics'):
    print("\n📋 Métricas por Clase:")
    print("-"*70)
    print(f"{'Clase':<25} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print("-"*70)
    for clase, m in metrics['class_metrics'].items():
        if isinstance(m, dict) and 'precision' in m:
            print(f"{clase:<25} {m['precision']:>10.3f} {m['recall']:>10.3f} {m['f1-score']:>10.3f} {int(m['support']):>10}")

---
## 📊 Paso 4: Generar Gráficas

In [ ]:
# Generar las 5 gráficas importantes
plots_params = {
    "plots_to_generate": [
        "feature_importance",
        "class_distribution",
        "confusion_matrix",
        "bmi_by_class",
        "correlation"
    ]
}

print("📊 Generando gráficas...")

result = api_call("POST", "/api/v1/nutritional/generate_plots", plots_params)

if "error" in result:
    print(f"❌ Error: {result['error']}")
else:
    print(f"✅ {result.get('message')}")
    print(f"   Gráficas generadas: {result.get('plots_generated', [])}")

In [ ]:
# Mostrar gráficas
if result.get('plots_base64'):
    for plot_name, b64_data in result['plots_base64'].items():
        print(f"\n{'='*50}")
        print(f"📊 {plot_name.upper().replace('_', ' ')}")
        print(f"{'='*50}")
        display(Image(data=base64.b64decode(b64_data)))

---
## 💾 Descargar Archivos (Opcional)

In [ ]:
# Descargar CSV de entrenamiento
# response = requests.get(f"{ML_API_URL}/api/v1/nutritional/download_csv")
# with open("training_data.csv", "wb") as f:
#     f.write(response.content)
# print("✅ CSV descargado: training_data.csv")

In [ ]:
# Descargar modelo entrenado
# response = requests.get(f"{ML_API_URL}/api/v1/nutritional/download_model")
# with open("nutritional_predictor.pkl", "wb") as f:
#     f.write(response.content)
# print("✅ Modelo descargado: nutritional_predictor.pkl")

---
## ✅ Resumen

El modelo de predicción nutricional ha sido entrenado exitosamente.

**Próximos pasos:**
1. El modelo está guardado en `models/nutritional_predictor.pkl`
2. El backend de Nutricion-api puede cargarlo para hacer predicciones
3. Las gráficas están en `plots/`